In [3]:
import setup
setup.init_django()

In [4]:
from blog.models import BlogPost

In [5]:
qs = BlogPost.objects.all().delete()

In [6]:
qs


(0, {})

In [7]:
docs = [
    "The dog jumped over the cat",
    "The cat jumped over the dog",
    "it is very warm today",
    "The cat is yellow and the dog is red"
]

In [8]:
new_data = []
for i, x in enumerate(docs):
    new_data.append(BlogPost(title=f"Blog Post {i+1}", content=x, can_delete=True))

BlogPost.objects.bulk_create(new_data)


[<BlogPost: BlogPost object (13)>,
 <BlogPost: BlogPost object (14)>,
 <BlogPost: BlogPost object (15)>,
 <BlogPost: BlogPost object (16)>]

In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/home/mousavi-m/Documents/talk-to-django/all-MiniLM-L6-v2")

/home/mousavi-m/.pyenv/versions/django_to_talk/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name /home/mousavi-m/Documents/talk-to-django/all-MiniLM-L6-v2. Creating a new one with mean pooling.


In [11]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [12]:
def get_embedding(text):
    embedding = model.encode([text])
    return embedding[0]

In [14]:
t = get_embedding("The dog jumped  the cat")
t

array([ 2.29751378e-01,  5.21904193e-02,  2.69422442e-01,  3.13673407e-01,
       -1.61949247e-01,  2.94956207e-01, -1.49297208e-01, -6.27667382e-02,
        7.16924593e-02, -1.44992217e-01,  3.55529100e-01,  3.83082889e-02,
        6.53436184e-02, -1.65470943e-01, -5.14907427e-02, -7.45399967e-02,
       -6.51313424e-01,  5.49471788e-02,  4.59080368e-01, -1.43820018e-01,
       -1.58220291e-01, -2.37345338e-01, -6.96481094e-02, -2.78592885e-01,
       -4.15547416e-02,  2.10807085e-01, -9.65539292e-02, -2.47809082e-01,
        7.54933730e-02, -7.43890628e-02, -1.72067389e-01, -7.30326176e-02,
       -2.79711902e-01,  2.66624361e-01, -2.23932952e-01, -3.93601686e-01,
        3.70261699e-01, -1.38748705e-01,  3.73328120e-01,  7.04822063e-01,
       -2.94373035e-01, -1.84417352e-01, -1.05393358e-01,  3.40771303e-02,
       -1.31664485e-01,  3.60766262e-01,  7.14174733e-02, -1.61552057e-01,
        7.26578385e-02,  7.88715258e-02, -1.53552532e-01,  4.95292217e-01,
        6.04516603e-02, -

In [15]:
qs = BlogPost.objects.filter(can_delete=True)
for post in qs:
    post.embedding = get_embedding(post.content)
    post.save() 

In [16]:
query = "The dog jumped over the cat"
query_embedding = get_embedding(query)

In [19]:
from pgvector.django import CosineDistance
from django.db.models import F

qs = BlogPost.objects.annotate(
    distance=CosineDistance('embedding', query_embedding),
    simarity=1 - F('distance')
).order_by('-simarity')

for obj in qs:
    print(obj.content, obj.distance, obj.simarity * 100)


The dog jumped over the cat 0.0 100.0
The cat jumped over the dog 0.01045568189901247 98.95443181009875
The cat is yellow and the dog is red 0.5003317445605779 49.96682554394221
it is very warm today 1.027799389600016 -2.7799389600015934
